In [ ]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from datetime import datetime
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
data=pd.read_csv(r"C:\Users\Rinshu\Downloads\Resume.csv")

In [3]:
data.head(5)

,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [4]:
data["Resume_html"][0]

'<div class="fontsize fontface vmargins hmargins linespacing pagesize" id="document"> <div class="section firstsection" id="SECTION_NAME500375979" style="\n      padding-top:0px;\n    "> <div class="paragraph PARAGRAPH_NAME firstparagraph" id="PARAGRAPH_500375979_1_326506904" style="\n      padding-top:0px;\n    "> <div class="name" itemprop="name"> <span class="field fName" id="500375979FNAM1"> </span> <span> </span> <span class="field" id="500375979LNAM1"> HR ADMINISTRATOR/MARKETING ASSOCIATE\n\nHR ADMINISTRATOR</span> </div> </div> </div> <div class="section" id="SECTION_SUMM500375981" style="\n      padding-top:0px;\n    "> <div class="heading bottomborder"> <div class="sectiontitle" id="SECTNAME_SUMM500375981"> Summary</div> </div> <div class="paragraph firstparagraph" id="PARAGRAPH_500375981_1_326506917" style="\n      padding-top:0px;\n    "> <div class="field singlecolumn" id="500375981FRFM1"> <p align="LEFT"> Dedicated Customer Service Manager with 15+ years of experience in H

In [5]:
data["Resume_str"][2276]

'         8TH GRADE LANGUAGE ARTS TEACHER       Summary      Teacher with excellent communication skills. Organized and driven with the innate ability to stay on task. Uses effective and efficient methods of teaching while focusing on the individual needs of each student. Seeking a position that will be both challenging and fulfilling.         Highlights           Lesson planning expertise    Academic performance evaluations  IEP familiarity  504 familiarity  Behavioral disorders knowledge       Certified Student Teacher Trainer    Tutoring experience    MS Office proficient    Standardized testing    Google Drive familiarity             Accomplishments      Achieved high growth on 2013-2014 school year End of Grade Assessment for Reading.   Chosen to be an assessment creator for the North Learning Community in Charlotte Mecklenburg Schools, based on high growth for 2013-2014 End of Grade assessment scores.   Helped more than 75%\xa0students reach their Individual Education Program goa

In [6]:
data.shape

(2484, 4)

In [7]:
data=data.drop(columns=["Resume_html"])

EXPERIENCE COLUMN

In [8]:
patterns = [
        r'(\d+)\+?\s*(?:years?|yrs?)\s*(?:of)?\s*experience',
        r'experience\s*(?:of|:|-)?\s*(\d+)\+?\s*(?:years?|yrs?)',
        r'(\d+)\+?\s*(?:years?|yrs?)\s*(?:in|as|with)',
        r'total\s*experience\s*[:\-]?\s*(\d+)\+?\s*(?:years?|yrs?)',
        r'(\d+)\s*-\s*\d+\s*(?:years?|yrs?)', 
        r'over\s*(\d+)\s*(?:years?|yrs?)',
        r'(\d+)\+?\s*(?:years?|yrs?)'
    ]
def information(text):
    for p in patterns:
       match=re.search(p,text)
       if match:
           return int(match.group(1))
    return 0     

In [9]:
MONTH_NAMES = r'(?:january|february|march|april|may|june|july|august|september|october|november|december|jan|feb|mar|apr|jun|jul|aug|sep|sept|oct|nov|dec)'
data_pattern= re.compile(rf'(?:{MONTH_NAMES}\s+|\d{{1,2}}/)?(\d{{4}})\s*(?:-|–|to)\s*(?:{MONTH_NAMES}\s+|\d{{1,2}}/)?(present|current|\d{{4}})',re.IGNORECASE)
def extract_experience(text):
    match=re.findall(data_pattern,text.lower())
    total_year=0
    for start,end in match:
        start_year=int(start)
        end=end.strip()
        if 'present' in end or 'current' in end:
            end_year = datetime.now().year
        else:
            try:
                end_year = int(end)
            except ValueError:
                continue 
        total_year+=end_year-start_year
    return total_year    

In [10]:
def combine_exp(text):
    exp=information(text)
    if exp==0:
        exp=extract_experience(text)
    return exp    

In [11]:
data["experience"]=data["Resume_str"].apply(combine_exp)

In [12]:
sample_text = data[data['ID'] == 12413512]['Resume_str'].values[0]
matches = data_pattern.findall(sample_text.lower())
print(matches)

[('2013', '2014'), ('2013', '2014'), ('2013', 'current'), ('2013', '2016'), ('2015', 'current'), ('2015', 'current'), ('2015', 'current'), ('2015', 'current'), ('2015', 'current'), ('2014', '2014'), ('2014', '2014')]


In [13]:
data.sample(5)

,ID,Resume_str,Category,experience
2143,22754014,CONTENT STRATEGIST Summary ...,PUBLIC-RELATIONS,41
2398,94137171,PLANT FULFILLMENT LEADER Summ...,AVIATION,18
578,10501991,BUSINESS DEVELOPMENT REPRESENTATIVE ...,BUSINESS-DEVELOPMENT,37
267,13405733,DIRECTOR OF INFORMATION TECHNOLOGY ...,INFORMATION-TECHNOLOGY,13
1765,30542184,CONSTRUCTION ENGINEERING SUPERVISOR ...,ENGINEERING,20


TEXT PREPROCESSING

In [14]:
nltk.download("stopwords")
nltk.download("wordnet")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Rinshu\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Rinshu\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [17]:
stop_words=set(stopwords.words("english"))
lemmantize=WordNetLemmatizer()

In [18]:
def text_preprocessing(text):
    text=text.lower()
    text=re.sub(r'[^a-zA-Z\s]',"",text)
    text=re.sub(r'https?://\S+',"",text)
    word=text.split()
    words=[lemmantize.lemmatize(w) for w in word if w not in stop_words]
    return " ".join(words)

In [19]:
data["clean_resume_str"]=data["Resume_str"].apply(text_preprocessing)

In [20]:
data["clean_resume_str"][0]

'hr administratormarketing associate hr administrator summary dedicated customer service manager year experience hospitality customer service management respected builder leader customerfocused team strives instill shared enthusiastic commitment customer service highlight focused customer satisfaction team management marketing savvy conflict resolution technique training development skilled multitasker client relation specialist accomplishment missouri dot supervisor training certification certified ihg customer loyalty marketing segment hilton worldwide general manager training certification accomplished trainer cross server hospitality system hilton onq micros opera pm fidelio opera reservation system or holidex completed course seminar customer service sale strategy inventory control loss prevention safety time management leadership performance assessment experience hr administratormarketing associate hr administrator dec current company name city state help develop policy directs c

TF IDF

In [21]:
tf=TfidfVectorizer()
job_desc=input("enter your requirements:")
cleaned_job_des=text_preprocessing(job_desc)
cleaned_jd=data["clean_resume_str"]
corpus=[cleaned_job_des] + cleaned_jd.tolist()

result=tf.fit_transform(corpus).toarray()

In [22]:
result

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [23]:
jd_vector=result[0].reshape(1,-1)
clean_vector=result[1:]

similarity=cosine_similarity(jd_vector,clean_vector).flatten()

In [26]:
data["similarity"]=similarity*100
top_matches=data.sort_values(by="similarity",ascending=False).head(10)
top_matches.head(3)


,ID,Resume_str,Category,experience,clean_resume_str,similarity
22,25676643,HR SPECIALIST Summary An Hum...,HR,9,hr specialist summary human resource specialis...,5.369108
40,41523474,HR EXECUTIVE Summary Dual s...,HR,1,hr executive summary dual specialization domai...,3.971755
2026,32773331,PROJECT ENGINEER & PROJECT MANAGER ...,CONSTRUCTION,20,project engineer project manager summary const...,3.811200
